# Lesson 5: MCP and A2A

For each cell: **guess what it will print first**, then press ▶ and check.

From [Build an AI Team](https://gradlabs.in/learn/agent-team/lessons/lesson5.html).

In [ ]:
import json   # JSON: text that holds lists and dictionaries, so programs can send them to each other

In [ ]:
# An MCP server for the order sheet. It lists its tools, and runs one when asked.
ORDERS = {101: "red frock, size 5, paid", 102: "pastel frock, size 3, packed"}

def order_status(args):
    return ORDERS.get(args["order_id"], "No order with that number")

order_server = {
    "tools": [{"name": "order_status",
               "description": "Look up one order by its number.",
               "inputSchema": {"type": "object",
                               "properties": {"order_id": {"type": "integer"}},
                               "required": ["order_id"]}}],
    "run": {"order_status": order_status},
}

In [ ]:
# tools/list: the first thing a client asks
def list_tools(server):
    return {"jsonrpc": "2.0", "id": 1, "result": {"tools": server["tools"]}}

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The client turns the list into lines for the bot's instructions
def describe(server):
    lines = []
    for t in list_tools(server)["result"]["tools"]:
        needs = ", ".join(t["inputSchema"]["required"])
        lines.append(f'{t["name"]}({needs}): {t["description"]}')
    return "\n".join(lines)

print(describe(order_server))

In [ ]:
# tools/call: run one tool with its arguments, or send back an error
def call_tool(server, request):
    name = request["params"]["name"]
    args = request["params"]["arguments"]
    tool = None
    for t in server["tools"]:
        if t["name"] == name:
            tool = t
    if tool is None:
        return {"jsonrpc": "2.0", "id": request["id"], "error": {"message": "Unknown tool: " + name}}
    for need in tool["inputSchema"]["required"]:
        if need not in args:
            return {"jsonrpc": "2.0", "id": request["id"], "error": {"message": "Missing: " + need}}
    text = server["run"][name](args)
    return {"jsonrpc": "2.0", "id": request["id"], "result": {"content": [{"type": "text", "text": text}]}}

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# A bot asks for a tool by writing one line: CALL, then JSON
def handle(line, server):
    try:
        call = json.loads(line.replace("CALL ", "", 1))
    except Exception:
        return "Error: not valid JSON"
    request = {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": call}
    reply = call_tool(server, request)
    if "error" in reply:
        return "Error: " + reply["error"]["message"]
    return reply["result"]["content"][0]["text"]

print(handle('CALL {"name": "order_status", "arguments": {"order_id": 101}}', order_server))
print(handle('CALL {"name": "order_status", "arguments": {"id": 101}}', order_server))

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# A2A: another company's agent. First, read its Agent Card: who it is, what it can do.
courier_card = {"name": "SwiftShip Courier Agent",
                "url": "https://swiftship.example/a2a",
                "skills": [{"id": "book_pickup", "description": "Book a parcel pickup"}]}
print(courier_card["name"], "can:", courier_card["skills"][0]["description"])

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Send it a task. It works on its own side; we watch the task's status change.
def send_task(card, message):
    task = {"id": "t-7", "status": "submitted", "message": message}
    print(task["status"])
    for status in ["working", "completed"]:   # what the courier's agent reports back
        task["status"] = status
        print(task["status"])
    task["result"] = "Pickup booked: Saturday 4 PM, 6 parcels"
    return task

task = send_task(courier_card, "Pick up 6 parcels from Kochi on Saturday.")
print(task["result"])

## Now a real LLM

Now with a small free model (Qwen2.5 1.5B) writing the replies.
The first run downloads it (about 3 GB), so give it a few minutes.

Small models make mistakes. The program checks everything they write, and its limits stop a chat that goes wrong.

In [ ]:
# A real LLM picks the tool: a small, free model that runs inside Colab
# (faster with a GPU: Runtime → Change runtime type → T4 GPU)
import torch                        # the library that does the model's math
from transformers import pipeline   # a ready-made way to load a model and use it

In [ ]:
# Load Qwen2.5 1.5B (about 3 GB, downloaded the first time)
model = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                 device=0 if torch.cuda.is_available() else -1)

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The bot's instructions: its role, then the tool list from the MCP server (tools/list)
INSTRUCTIONS = ("You are Operations at Amal Designers. You can use these tools:\n"
                + describe(order_server) + "\n"
                + 'To use a tool, write one line: CALL {"name": "tool_name", "arguments": {...}}\n'
                + "Write only that line.")

def ask(question):
    text = INSTRUCTIONS + "\n\nQuestion: " + question
    out = model([{"role": "user", "content": text}], max_new_tokens=60, do_sample=False)
    reply = out[0]["generated_text"][-1]["content"]
    print("LLM:", reply)
    for line in reply.split("\n"):
        if line.strip().startswith("CALL "):
            return handle(line.strip(), order_server)   # the program checks and runs it
    return "Error: no CALL line"

print(ask("Where is order 101?"))
print(ask("Has order 102 been packed?"))
print(ask("What about order 999?"))   # not in the sheet: the tool says so